In [7]:
# Retrivers is component in langchain that fetches relevant docu from data sourcs in responce to query.
# fucntion takes, quvery and give langchain document object
# All Retrivers are Runnable in Langchain

# Types of Retrivers
    # 1) Based on Data Source
        # a) Wikipedia retievrs
        # b) Youtbe retievrs
        # c) vector store retievers
        # d) Archive retrievers
        
    # 2) Based on search stragery
        # a) NMP 
        # b) Multiquery

In [8]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from typing import TypedDict , Annotated , List , Optional
import warnings
warnings.filterwarnings("ignore")
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser , JsonOutputParser 

load_dotenv()
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

parser = StrOutputParser()

llm_gemini = ChatGoogleGenerativeAI(model="gemini-2.0-flash" , api_key= GOOGLE_API_KEY)
llm_gemini.invoke("who is father of india").content

'Mahatma Gandhi is widely considered the "Father of the Nation" in India.'

### Retrievers based on Data Type

In [9]:
# Text Loader
# transcipts of videos logs
import warnings
from langchain_community.document_loaders import TextLoader
warnings.filterwarnings("ignore")

loader = TextLoader(file_path= "Data\sample_text.txt" , encoding= "utf-8")

docs = loader.load()
docs = str(docs)

print(docs)
print((docs.__class__))

[Document(metadata={'source': 'Data\\sample_text.txt'}, page_content='Data Scientist: 🕵️\u200d♂️ Finds insights in data, builds models. AI Engineer: 🛠️ Deploys those models at scale!\n\nThink: Data Scientist is the architect, AI Engineer is the construction crew. Both build the future! 🚀 #DataScience #AI #Tech #Jobs #Career\n## Data Scientist vs. AI Engineer: Decoding the Buzz! 🤖🧠\n\nEver get these two roles mixed up? You\'re not alone! Data Scientist and AI Engineer are both hot careers, but they tackle different sides of the AI coin.\n\nThink of it this way: **Data Scientists are the visionaries. They *discover* insights from data, building models to predict future trends and answer critical business questions.** They\'re all about asking "why" and "what."\n\n**AI Engineers are the builders. They *deploy* those models into real-world applications.** They\'re focused on making the AI work, scale, and integrate seamlessly. Think production-ready AI!\n\n**🔑 Key Differences:**\n\n*   **D

In [10]:
# wikipedia retrievers
# query wikipedia api and retriever relevant articals
# perform searching so this is not document loder hence retriever
import warnings
warnings.filterwarnings("ignore")
import chroma_db
from langchain_community.retrievers import WikipediaRetriever

retriever = WikipediaRetriever(top_k_results= 3 , lang= "en")

docs = retriever.invoke("Impact of WWI on indian Economy")

for i , doc  in enumerate(docs):
    print(f"for artical {i} title is {doc.page_content[:500]}")

for artical 0 title is World War I or the First World War (28 July 1914 – 11 November 1918), also known as the Great War, was a global conflict between two coalitions: the Allies (or Entente) and the Central Powers. Main areas of conflict included Europe and the Middle East, as well as parts of Africa and the Asia-Pacific. There were important developments in weaponry including tanks, aircraft, artillery, machine guns, and chemical weapons. One of the deadliest conflicts in history, it resulted in an estimated 30 mill
for artical 1 title is The Indian Armed Forces are the military forces of the Republic of India. It consists of three professional uniformed services: the Indian Army, the Indian Navy, and the Indian Air Force. Additionally, the Indian Armed Forces are supported by the Central Armed Police Forces, the Indian Coast Guard, and  the Special Frontier Force and various inter-service commands and institutions such as the Strategic Forces Command, the Andaman and Nicobar Command

In [ ]:
# Vector Store Retrievers
# Most common, very famous
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings

doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )

final_docu = [doc1 , doc2 , doc3 , doc4 , doc5]

# Embedding Model
embedding_model = GoogleGenerativeAIEmbeddings(model='models/gemini-embedding-001')

vector_store = Chroma.from_documents(
    documents= final_docu,
    embedding= embedding_model,
    collection_name="my_collection"
)


### NMR ( for diversity so that we don't have same results )

In [12]:
# Retrivers give us various staegory to seach as oppose to Venilia seach in vector store

# 1) NMR (Maxmial Marginal Relevance)
# What if my result are similar that retrivers Fetch
# Results are similar to query but diff from each other 
# Similar to query but not similar to each other 
# Better for diversity

In [13]:
# Sample Use
documents = [
    Document(page_content="LangChain helps developers build LLM applications easily."),
    Document(page_content="Chroma is a vector database optimized for LLM-based search."),
    Document(page_content="Embeddings convert text into high-dimensional vectors."),
    Document(page_content="OpenAI provides powerful embedding models."),
]

vector_store = Chroma.from_documents(
    documents= documents, 
    embedding= embedding_model,
    collection_name= "chroma_db"
)

In [19]:
retriver = vector_store.as_retriever(
    search_type  = 'mmr',
    search_kwargs = {"k" : 3 , "lambda_mult" :0}
)

retriver.invoke("what is Langchain")

[Document(metadata={}, page_content='LangChain helps developers build LLM applications easily.'),
 Document(metadata={}, page_content='Chroma is a vector database optimized for LLM-based search.'),
 Document(metadata={}, page_content='Embeddings convert text into high-dimensional vectors.')]

### **Multi Query Retriver**( What is query is not clear enough and vauge)

In [ ]:
# A single query is use to generate mutiple query 
# generated Query are less ambigous but revelant
# send all query and do vector seach and do reranking

In [21]:
# Sample Code
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

vector_store = Chroma.from_documents(
    documents= all_docs,
    embedding= embedding_model,
    collection_name="chroma_db"
)

In [22]:
from langchain.retrievers.multi_query import MultiQueryRetriever

base_retriever = vector_store.as_retriever(
    search_type = "similarity",
    search_kwargs = {"k" : 3}
)
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever= base_retriever,
    llm= llm_gemini,
)

multi_query_retriever.invoke("How to live a healhty life?")

[Document(metadata={'source': 'H5'}, page_content='Drinking sufficient water throughout the day helps maintain metabolism and energy.'),
 Document(metadata={'source': 'H2'}, page_content='Consuming leafy greens and fruits helps detox the body and improve longevity.'),
 Document(metadata={'source': 'H4'}, page_content='Mindfulness and controlled breathing lower cortisol and improve mental clarity.'),
 Document(metadata={'source': 'H1'}, page_content='Regular walking boosts heart health and can reduce symptoms of depression.')]

In [23]:
base_retriever.invoke("How to live a healhty life?")

[Document(metadata={'source': 'H2'}, page_content='Consuming leafy greens and fruits helps detox the body and improve longevity.'),
 Document(metadata={'source': 'H5'}, page_content='Drinking sufficient water throughout the day helps maintain metabolism and energy.'),
 Document(metadata={'source': 'H1'}, page_content='Regular walking boosts heart health and can reduce symptoms of depression.')]